# 02 · Co-occurrence → 'things that go together'

The central signal for building playlists from owned playlists is
**co-occurrence**: which tracks (and artists) the user groups together.
This is exactly the signal behind playlist embeddings and Automatic Playlist
Continuation (APC).

Grounding: [`research/05`](../../research/05-inferring-taste-from-libraries.md)
(co-occurrence, APC, word2vec-style embeddings) and
[`research/01`](../../research/01-what-makes-a-successful-playlist.md) (cohesion).

## Setup

In [ ]:
import sqlite3
import pandas as pd

DB = '../playlists.db'  # run build_db.py first if this is missing
con = sqlite3.connect(DB)
def q(sql, **kw):
    return pd.read_sql_query(sql, con, params=kw)

q('SELECT COUNT(*) AS playlists FROM playlists')

## Artists that co-occur most
Raw co-occurrence weight = # playlists sharing both artists.

In [ ]:
pairs = q('''SELECT artist_a, artist_b, weight
             FROM artist_cooccurrence ORDER BY weight DESC LIMIT 20''')
pairs

## Raw counts favour popular artists — normalise with PMI
Pointwise Mutual Information down-weights artists that co-occur simply because
they're *everywhere*, surfacing genuinely associated pairs.

PMI(a,b) = log( P(a,b) / (P(a)·P(b)) ), estimated over playlists.

In [ ]:
# playlists each artist appears in (denominator for P(a))
art_pl = q('''SELECT t.artist, COUNT(DISTINCT pt.playlist_id) AS n
              FROM tracks t JOIN playlist_tracks pt ON pt.track_id=t.id
              GROUP BY t.artist''').set_index('artist')['n'].to_dict()
N = q('SELECT COUNT(*) AS n FROM playlists')['n'][0]
import math
def pmi(row):
    pa, pb = art_pl.get(row.artist_a,1)/N, art_pl.get(row.artist_b,1)/N
    pab = row.weight / N
    return math.log(pab/(pa*pb))
all_pairs = q('SELECT artist_a, artist_b, weight FROM artist_cooccurrence WHERE weight>=2')
all_pairs['pmi'] = all_pairs.apply(pmi, axis=1)
all_pairs.sort_values('pmi', ascending=False).head(20)

## 'More like this artist' — a minimal recommender
Given a seed artist, rank its co-occurring artists. This is the seed of a
candidate generator: expand a user's playlist by walking the co-occurrence graph.

In [ ]:
def neighbours(artist, k=10):
    sql = '''SELECT CASE WHEN artist_a=:a THEN artist_b ELSE artist_a END AS artist, weight
             FROM artist_cooccurrence WHERE artist_a=:a OR artist_b=:a
             ORDER BY weight DESC LIMIT :k'''
    return q(sql, a=artist, k=k)
neighbours('Arctic Monkeys')

## Toward embeddings
Co-occurrence counts are a sparse similarity matrix. The standard next step is
to learn dense **track/artist embeddings** (word2vec treating each playlist as a
'sentence' of tracks), giving smooth similarity and analogies. That needs
`gensim`/`sentence-transformers` (see `requirements.txt`); the co-occurrence graph
above is the honest, dependency-free starting point.

Next: [`03-flow-eval-and-llm-ideas.ipynb`](03-flow-eval-and-llm-ideas.ipynb).